# Resuming: MQTT + OD2000 Integration (`feat-mqtt-od2000-integration`)

**Written:** 2026-07-18  
**Branch:** `feat-mqtt-od2000-integration` (off `develop`)  
**Status:** All code written and tested offline. Nothing has been run on real hardware yet.

This notebook is a return-to-work guide. Work through the sections in order. Each section has runnable cells to verify a piece of the system before moving on.

---

## What was built

| File | Purpose |
|------|---------|
| `src/laguna/pi/gauge_publisher.py` | Standalone Pi script: reads Massa gauge over serial, publishes to `laguna/gauge/water_level_mm` |
| `src/laguna/pi/scan_runner.py` | Standalone Pi script: runs a full topographic scan — BLC serial commands AND OD2000 MQTT subscription, all Pi-local |
| `src/laguna/mqtt/__init__.py` | `MqttSubscriber` — paho-mqtt wrapper with per-topic buffering, laguna subsystem interface |
| `src/laguna/rangefinder/__init__.py` | `RangefinderSubsystem` + `decode_od2000_pdin()` |
| `src/laguna/robot/macron/profiler.py` | `TopographicProfiler` — SSH-orchestrates scan_runner.py, retrieves CSV |
| `docs/MQTT_AL1342_SETUP.md` | One-time hardware bring-up guide (AL1342 → Mosquitto → OD2000 subscribe) |
| `docs/subsystems/rangefinder.md` | Narrative article for the rangefinder subsystem |
| `tests/test_rangefinder.py` | 28 tests: PDIN decoding, JSON path, RangefinderSubsystem |
| `tests/test_mqtt_subscriber.py` | 30 tests: MqttSubscriber lifecycle and message buffering |
| `tests/test_scan_runner_logic.py` | 18 tests: BLC serial parsing + dead-reckoning math |
| `tests/test_profiler.py` | 29 tests: TopographicProfiler orchestration |

**Existing tests unaffected:** 289 total tests pass, including all 184 pre-existing tests.

---

## Network / hostname recap

The lab LAN is isolated (no internet). The laguna PC runs dnsmasq as DHCP+DNS server for the whole lab LAN.

| Device | Hostname | Notes |
|--------|----------|-------|
| Raspberry Pi | `red.lab` | Runs Mosquitto broker, `serial_bridge.py`, gantry agent |
| ifm AL1342 | `al1342.lab` | Needs a DHCP reservation + dnsmasq A-record |
| Laguna PC | (DNS server) | All lab devices resolve hostnames through here |

**Open question:** The AL1342 manual only shows IP addresses in MQTT callback URL examples (`mqtt://192.168.x.x:1883/...`). It is unconfirmed whether `mqtt://red.lab:1883/laguna/od2000` works. If hostname resolution fails in the AL1342's MQTT client, substitute the Pi's IP. Nothing else needs to change.

---

## Step 0 — Merge / branch check

In [ ]:
import subprocess

result = subprocess.run(
    ["git", "branch", "--show-current"],
    capture_output=True, text=True, cwd="/home/eric/mysoftware/laguna"
)
branch = result.stdout.strip()
print(f"Current branch: {branch}")
assert branch == "feat-mqtt-od2000-integration", (
    f"Expected feat-mqtt-od2000-integration, got {branch!r}. "
    "Run: git checkout feat-mqtt-od2000-integration"
)

In [ ]:
# Confirm tests still pass after any changes made since the branch was created
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-q", "--no-header", "--tb=short"],
    capture_output=True, text=True, cwd="/home/eric/mysoftware/laguna"
)
print(result.stdout[-3000:])  # last 3000 chars
if result.returncode != 0:
    print(result.stderr[-1000:])

---

## Step 1 — Install Mosquitto on the Pi

Do this once over SSH before anything else.

```bash
ssh oak@red.lab
sudo apt update && sudo apt install -y mosquitto mosquitto-clients
sudo systemctl enable mosquitto && sudo systemctl start mosquitto
```

Create `/etc/mosquitto/conf.d/laguna.conf`:
```
listener 1883
allow_anonymous true
log_dest file /var/log/mosquitto/mosquitto.log
```

Then `sudo systemctl restart mosquitto`.

**Verify:** The cell below SSHes to the Pi and checks Mosquitto status.

In [ ]:
import paramiko

PI_HOST = "red.lab"
PI_USER = "oak"
PI_KEY  = "/home/eric/.ssh/id_ed25519"

def ssh_run(cmd, host=PI_HOST, user=PI_USER, key=PI_KEY):
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    client.connect(host, username=user, key_filename=key)
    _, stdout, stderr = client.exec_command(cmd)
    out = stdout.read().decode()
    err = stderr.read().decode()
    client.close()
    return out, err

out, err = ssh_run("systemctl is-active mosquitto")
print("mosquitto:", out.strip())
assert out.strip() == "active", "Mosquitto is not running — follow Step 1 setup above"

---

## Step 2 — Assign AL1342 a hostname

The AL1342 needs a DHCP reservation and DNS A-record in dnsmasq on the laguna PC.

1. Find the AL1342's MAC address (label on the device, or check dnsmasq's DHCP lease file: `/var/lib/misc/dnsmasq.leases`)
2. Add to `/etc/dnsmasq.conf` (or a conf.d file):
   ```
   dhcp-host=<MAC>,al1342,192.168.X.Y
   address=/al1342.lab/192.168.X.Y
   ```
3. Restart dnsmasq: `sudo systemctl restart dnsmasq`
4. Power-cycle or renew the AL1342's DHCP lease

**Verify:**

In [ ]:
import socket

AL1342_HOST = "al1342.lab"

try:
    ip = socket.gethostbyname(AL1342_HOST)
    print(f"al1342.lab resolves to {ip} ✓")
except socket.gaierror as e:
    print(f"Cannot resolve al1342.lab: {e}")
    print("→ Follow Step 2 to add the DHCP reservation and A-record")

In [ ]:
# Check the IoT-Core Visualizer is reachable
import urllib.request

try:
    with urllib.request.urlopen(f"http://{AL1342_HOST}/web/subscribe", timeout=5) as resp:
        print(f"IoT-Core Visualizer HTTP {resp.status} ✓")
except Exception as e:
    print(f"Cannot reach AL1342 web UI: {e}")
    print("→ Check physical Ethernet connection and IP assignment")

---

## Step 3 — Confirm OD2000 on AL1342

Open `http://al1342.lab/web/subscribe` in a browser, go to **Parameter → Iolinkmaster**, and find the port the OD2000 is plugged into. Note the port number — you'll use it everywhere as `pdin_port`.

Expected fields:
- `vendorid` = 85  
- `productname` contains "OD2000"

**Set `PDIN_PORT` here and carry it through the rest of the notebook:**

In [ ]:
PDIN_PORT = 1  # ← CHANGE THIS to whichever port the OD2000 is on (1–8)
print(f"OD2000 is on IO-Link port {PDIN_PORT}")

---

## Step 4 — Configure AL1342 MQTT command channel

The AL1342 can't receive MQTT commands until MQTT is configured. Use the IoT-Core Visualizer's **Notification** tab wizard, or run the cells below to POST bootstrap commands over HTTP.

> **Hostname vs. IP:** if the `brokerIP` field rejects `red.lab`, change `BROKER_ADDR` below to the Pi's actual IP address. Nothing else needs to change.

See `docs/MQTT_AL1342_SETUP.md` § Steps 3–4 for the full manual walkthrough.

In [ ]:
import json, urllib.request

BROKER_ADDR = "red.lab"   # ← substitute Pi IP if hostname doesn't work in AL1342
AL1342_URL  = f"http://{AL1342_HOST}/iolinkmaster"

def al1342_post(payload: dict) -> dict:
    data = json.dumps(payload).encode()
    req = urllib.request.Request(AL1342_URL, data=data,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as resp:
        return json.loads(resp.read())

bootstrap_cmds = [
    {"code": "request", "cid": 1,
     "adr": "/connections/mqttConnection/MQTTSetup/mqttCmdChannel/status/start"},
    {"code": "request", "cid": 2,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/brokerIP/setdata",
     "data": {"newvalue": BROKER_ADDR}},
    {"code": "request", "cid": 3,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/brokerPort/setdata",
     "data": {"newvalue": "1883"}},
    {"code": "request", "cid": 4,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/cmdTopic/setdata",
     "data": {"newvalue": "laguna/al1342/cmd"}},
    {"code": "request", "cid": 5,
     "adr": "/connections/mqttConnection/mqttCmdChannel/mqttCmdChannelSetup/defaultReplyTopic/setdata",
     "data": {"newvalue": "laguna/al1342/reply"}},
]

for cmd in bootstrap_cmds:
    try:
        resp = al1342_post(cmd)
        code = resp.get("code", "?")
        print(f"cid={cmd['cid']} → {code}")
    except Exception as e:
        print(f"cid={cmd['cid']} FAILED: {e}")

---

## Step 5 — Subscribe AL1342 to OD2000 cyclic data

This tells the AL1342 to start publishing OD2000 PDIN data to `laguna/od2000` every 100 ms.

In [ ]:
import paho.mqtt.client as mqtt
import threading, queue, time

BROKER = "red.lab"
OD2000_TOPIC = "laguna/od2000"
AL1342_CMD   = "laguna/al1342/cmd"
AL1342_REPLY = "laguna/al1342/reply"

# Subscribe request — note: replace [N] with actual port number
callback_url = f"mqtt://{BROKER_ADDR}:1883/{OD2000_TOPIC}"
pdin_path    = f"/iolinkmaster/port[{PDIN_PORT}]/iolinkdevice/pdin"

subscribe_cmd = {
    "code": "request", "cid": 10,
    "adr": "/timer[1]/counter/datachanged/subscribe",
    "data": {
        "callback": callback_url,
        "datatosend": [pdin_path],
    }
}
interval_cmd = {
    "code": "request", "cid": 11,
    "adr": "/timer[1]/interval/setdata",
    "data": {"newvalue": 100}   # 100 ms = 10 Hz starting point
}

print(f"Callback URL: {callback_url}")
print(f"PDIN path:    {pdin_path}")

reply_q = queue.Queue()

def on_reply(client, userdata, msg):
    reply_q.put(json.loads(msg.payload))

c = mqtt.Client(client_id="laguna_setup")
c.connect(BROKER, 1883)
c.subscribe(AL1342_REPLY)
c.on_message = on_reply
c.loop_start()

time.sleep(0.5)  # let subscription register

for cmd in [subscribe_cmd, interval_cmd]:
    c.publish(AL1342_CMD, json.dumps(cmd))
    try:
        reply = reply_q.get(timeout=5)
        print(f"cid={cmd['cid']} reply code: {reply.get('code', '?')}")
    except queue.Empty:
        print(f"cid={cmd['cid']} — no reply (command channel may not be configured yet)")

c.loop_stop()
c.disconnect()

---

## Step 6 — Verify OD2000 data stream

Collect 10 seconds of MQTT messages and decode the PDIN payload. If this works, the full data pipeline is alive.

In [ ]:
import sys
sys.path.insert(0, "/home/eric/mysoftware/laguna/src")

from laguna.rangefinder import decode_od2000_pdin

collected = []
collect_done = threading.Event()

def on_od2000(client, userdata, msg):
    wall_time = time.time()
    try:
        payload = json.loads(msg.payload)
        pdin_key = f"/iolinkmaster/port[{PDIN_PORT}]/iolinkdevice/pdin"
        hex_str  = payload["data"]["payload"][pdin_key]["data"]
        decoded  = decode_od2000_pdin(hex_str)
        collected.append({"wall_time": wall_time, "hex": hex_str, **decoded})
    except Exception as e:
        collected.append({"wall_time": wall_time, "error": str(e), "raw": msg.payload[:200]})

c = mqtt.Client(client_id="laguna_verify")
c.connect(BROKER, 1883)
c.subscribe(OD2000_TOPIC)
c.on_message = on_od2000
c.loop_start()

print("Collecting for 10 s …")
time.sleep(10)
c.loop_stop()
c.disconnect()

ok     = [s for s in collected if "error" not in s]
errors = [s for s in collected if "error" in s]

print(f"\nTotal messages : {len(collected)}")
print(f"Decoded OK     : {len(ok)}")
print(f"Decode errors  : {len(errors)}")

if ok:
    rate = len(ok) / 10.0
    dists = [s["distance_mm"] for s in ok]
    print(f"\nAchieved rate  : {rate:.1f} Hz")
    print(f"Distance range : {min(dists):.1f} – {max(dists):.1f} mm")
    print(f"Latest sample  : {ok[-1]}")
else:
    print("\nNo OK samples. Check:")
    print("  1. AL1342 command channel configured (Step 4)?")
    print("  2. OD2000 subscribe sent (Step 5)?")
    print("  3. PDIN_PORT correct?")
    if errors:
        print(f"\nFirst error: {errors[0]}")

### What the distance values should look like

- **Plausible range:** 20–1200 mm (OD2000 measurement range)
- **If distance_mm ~ 0.0002 when expecting ~200 mm:** unit is wrong — AL1342 may publish µm not nm; divide by 1000 instead of 1,000,000
- **If distance_mm looks wildly wrong (e.g. negative or > 10,000):** check byte order — PDIN may be little-endian not big-endian. To test: `int.from_bytes(bytes.fromhex(hex_str)[0:4], 'little', signed=True) / 1e6`
- **If `pdin_key` KeyError:** `PDIN_PORT` is wrong, or the path format differs from the manual spec

---

## Step 7 — Check serial_bridge.py port behavior

Before running a scan, confirm whether `serial_bridge.py` holds the BLC serial port permanently or only while a TCP client is connected. The scan requires exclusive access to that port.

See `docs/MQTT_AL1342_SETUP.md` § "serial_bridge.py Port Conflict" for the full explanation and workaround options.

In [ ]:
# Check whether serial_bridge.py currently holds the serial port open
out, err = ssh_run("ls -la /proc/$(pgrep -f serial_bridge.py)/fd 2>/dev/null | grep tty || echo 'not running or no tty fd'")
print(out or err)
print()
print("If you see a /dev/ttyUSB* or /dev/serial/ path: serial_bridge.py holds the port")
print("permanently. You need the SIGSTOP workaround before scanning (see MQTT_AL1342_SETUP.md).")
print()
print("If 'not running' or no tty fd: the port is opened lazily — no extra steps needed.")

---

## Step 8 — Run a topographic scan

Once Steps 1–7 are confirmed, you can run a real scan. The `TopographicProfiler` will:
1. Disconnect `gantry_agent.py` to release the BLC serial port
2. Deploy `scan_runner.py` to the Pi via SFTP
3. SSH-launch `scan_runner.py` which handles all BLC serial + MQTT locally
4. Wait for the move to complete and retrieve the CSV
5. Reconnect `gantry_agent.py`

> **Before running:** move the gantry to a safe starting position manually. `scan_runner.py` will read the current position as `start_pos_mm` and move to `end_mm` at `feed_rate_mm_s`.

In [ ]:
import sys
sys.path.insert(0, "/home/eric/mysoftware/laguna/src")

from laguna.config import Config
from laguna.robot.macron import GantryController
from laguna.robot.macron.profiler import TopographicProfiler

config = Config("config/example_config.yaml")
gantry = GantryController.from_config(config.get("gantry"))
gantry.connection.connect()

profiler = TopographicProfiler(
    gantry=gantry,
    pi_host="red.lab",
    pi_user="oak",
    pi_key="/home/eric/.ssh/id_ed25519",
    serial_device=config.get("gantry")["remote_serial_device"],
    pdin_port=PDIN_PORT,
    od2000_topic="laguna/od2000",
    output_dir="/tmp/laguna_profiles",
)

print("Profiler ready. Edit and run the next cell to start a scan.")

In [ ]:
# Adjust these before running
SCAN_AXIS        = "A1"    # X axis = A1, Y axis = A2
SCAN_END_MM      = 300.0   # target position in mm (absolute)
SCAN_RATE_MM_S   = 5.0     # slew speed — determines spatial resolution

result = profiler.scan(axis=SCAN_AXIS, end_mm=SCAN_END_MM, feed_rate_mm_s=SCAN_RATE_MM_S)

print(f"Samples collected : {result.metadata['samples']}")
print(f"Achieved rate     : {result.metadata.get('achieved_rate_hz', '?'):.1f} Hz")
print(f"Start position    : {result.metadata['actual_start_mm']:.2f} mm")
print(f"End position      : {result.metadata['actual_end_mm']:.2f} mm")
print(f"Actual distance   : {result.metadata['actual_distance_mm']:.2f} mm")
print(f"CSV path          : {result.path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = result.df
slew = df[df["in_ramp"] == 0]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(slew["pos_mm"], slew["distance_mm"], linewidth=0.8, label="slew phase")
ax.plot(df[df["in_ramp"]==1]["pos_mm"], df[df["in_ramp"]==1]["distance_mm"],
        ".", alpha=0.3, markersize=3, label="ramp phase (excluded)")
ax.set_xlabel("Gantry position (mm)")
ax.set_ylabel("OD2000 distance (mm)")
ax.set_title(f"Topographic profile — {SCAN_AXIS} axis")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nSlew-phase samples : {len(slew)}")
print(f"Spatial resolution : {SCAN_RATE_MM_S / result.metadata.get('achieved_rate_hz', 10):.2f} mm/sample")

---

## Open items and decision points

These need hardware to resolve. Document findings in the cells below as you go.

### OD2000 PDIN byte layout
The decoder assumes big-endian int32 distance in **nm** (bytes 0–3). Confirm against an LR DEVICE readout at a known distance.

| Finding | Action |
|---------|--------|
| `distance_mm` matches known physical distance | ✓ decoder is correct |
| `distance_mm` × 1000 matches known distance | AL1342 publishes µm — divide nm by 1000 |
| Distance is wildly wrong | Check byte order — try little-endian |

### AL1342 timer interval minimum
The manual lists 500 ms as minimum timer interval in one place, but the Step 5 cell sets 100 ms. If the AL1342 rejects 100 ms (reply code ≠ 200), try 500 ms first.

### Hostname in callback URL
If Step 5 fails with no reply, try substituting the Pi's IP address for `red.lab` in `BROKER_ADDR` and re-running Steps 4–5.

### serial_bridge.py port holding
Document result of Step 7 here once checked.

In [ ]:
# Fill in findings as you go
findings = {
    "pdin_byte_order": None,           # 'big-endian-nm', 'little-endian-nm', 'big-endian-um', ...
    "al1342_min_interval_ms": None,    # 100, 500, ...
    "callback_url_hostname_works": None, # True/False
    "serial_bridge_holds_port": None,  # True/False
    "achieved_mqtt_rate_hz": None,     # measured from Step 6
}
print(findings)

---

## NFS vs SFTP note

Currently, `TopographicProfiler` uses SFTP to both deploy `scan_runner.py` and retrieve the result CSV. If you set up an NFS mount of the Pi's output directory on the laguna PC, the retrieval step can be skipped — the CSV appears as a local file automatically.

To use NFS: add `nfs_output_path="/mnt/pi/profiles"` to the `TopographicProfiler` constructor, and update `profiler.py` to write to that path instead of SFTP-fetching. The deploy step (SFTP `put`) is unaffected — it writes `scan_runner.py` to `/tmp` on the Pi.

This has not been implemented yet — add it once the NFS mount is set up.

---

## What to merge when done

Branch `feat-mqtt-od2000-integration` → `develop`.

Before merging:
- [ ] All 289 tests still pass
- [ ] Steps 1–6 above verified on hardware
- [ ] `findings` dict above filled in and any decoder/config fixes committed
- [ ] `docs/MQTT_AL1342_SETUP.md` updated with confirmed hostname behavior and actual timer interval
- [ ] `docs/RANGEFINDER_PROFILING.md` open items updated with measured MQTT rate
- [ ] At least one real profile CSV in the repo or linked from the docs as a reference

After merging, open items for follow-up branches:
- Approach B: wire OD2000 Q2/Qa → INB 7 for hardware-triggered capture-latch (higher positional fidelity)
- MQTT auth: add Mosquitto credentials if the lab network ever changes
- NFS output directory if preferred over SFTP retrieval
- Gauge publisher wiring: deploy `gauge_publisher.py` as a persistent service rather than on-demand